# 1: Control case: FHISTC_LTso

In this exercise, you will create and run a control simulation using the `FHISTC_LTso` compset.

The `FHISTC_LTso` compset uses **prescribed historical** sea surface temperatures (SSTs), sea ice, and historical forcing data with the **low-top** atmosphere configuration. 

You will also customize CAM history output to produce 3-hourly instantaneous fields interpolated on a lat/lon grid that can later be used to compare against sensitivity experiments.


<div class="alert alert-secondary" style="text-align: left;">
`FHISTC_LTso` follows the AMIP protocol which is a standardized experimental protocol for global Atmospheric General Circulation Models (AGCMs).
    
For more information about the [AMIP protocol](https://pcmdi.llnl.gov/mips/amip/) and the [HadSST data sets](https://climatedataguide.ucar.edu/climate-data/sst-data-hadisst-v11)
</div>

**Why This Control Case Matters**

This control simulation will serve as **the control for later experiments in this section**. By keeping the compset, resolution, run length, and output variables consistent, you will be able to compare sensitivity experiments directly against this control case.

For example, later experiments modify SSTs, forcing files, datasets, or model parameters. The control case allows you to identify which differences are caused by the experiment itself.

<div class="alert alert-info">
<strong>Exercise: Customize your CAM history files</strong><br><br>

Create, configure, build, and run a case called **fhist.control** using:
- Compset: ``FHISTC_LTso``
- Resolution: ``ne16pg3_ne16pg3_mg17``

Run the case for **5 days** with **3-hourly instantaneous output** for the following variables:

`TS, PS, Z500, U850, U200, T850, T500, T200, CLDLOW,
PRECT, LHFLX, SHFLX, FLNT, FLNS`
You are also welcome to output additional variables of interest.

In addition, configure the history files to be automatically **interpolated to a regular 2° × 2° latitude-longitude grid**.

To improve throughput (how fast the model runs), change the task layout before running `case.setup`:

```bash
./xmlchange NTASKS=-6
```

**About the resolution for FHISTC_LTso**
- Supported resolution for this compset is `ne30pg3_ne30pg3_mg17`.
This is approximately a 1-degree atmosphere and land grid.

- During the tutorial, we recommend using `ne16pg3_ne16pg3_mg17`. This lower-resolution configuration runs faster and is better suited for short tutorial exercises. However, this resolution is not supported at this time and you will be required to add the flag `--run-unsupported`
</div>


<div class="alert alert-warning">  
<details>

<summary> <font face="Times New Roman" color='blue'>Click here for hints</font> </summary>
<br>

**How do I output 3 hourly instantaneous variables?**

- Use namelist variables: ``nhtfrq``, ``mfilt``, ``fincl``. 
- For more information, look at the chapter: <br>
**NAMELIST MODIFICATIONS** -> **Customize CAM output**


**What if I get an unsupported-grid warning?**

You might be getting this error if the `ne16pg3_ne16pg3_mg17` is not a supported resolution in the version of the model you are using. 

Overide this error by adding ``--run-unsupported`` to the create_newcase command


**Increase model throughput**

To improve throughput (how fast the model runs), change the task layout before running `case.setup`:

```bash
./xmlchange NTASKS=-6
```


</details>
</div>



<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>
 
 
**# Set environment variables** 

Set environment variables with the commands:
    
**For tcsh users** 
    
```
set CASENAME=fhist.control
set CASEDIR=/glade/u/home/$USER/cases/$CASENAME
set RUNDIR=/glade/derecho/scratch/$USER/$CASENAME/run
set COMPSET=FHISTC_LTso
set RESOLUTION=ne16pg3_ne16pg3_mg17
```
**For bash users** 
    
```
export CASENAME=fhist.control
export CASEDIR=/glade/u/home/$USER/cases/$CASENAME
export RUNDIR=/glade/derecho/scratch/$USER/$CASENAME/run
export COMPSET=FHISTC_LTso
export RESOLUTION=ne16pg3_ne16pg3_mg17
```

**# Make a case directory**

If needed create a directory `cases` into your home directory:
    
```
mkdir /glade/u/home/$USER/cases/
```
    

**# Create a new case**

Create a new case with the command ``create_newcase``:
```
cd /glade/u/home/$USER/code/my_cesm_code/cime/scripts/
./create_newcase --case $CASEDIR --res $RESOLUTION --compset $COMPSET   --run-unsupported
```

**# Change the job queue and account number**

If needed, change ``job queue`` and ``account number``.<br>
For instance, to run in the queue ``tutorial`` and the project number ``UESM0016``. You should use the project number given for this tutorial.
```  
cd $CASEDIR
./xmlchange JOB_QUEUE=tutorial,PROJECT=UESM0016 --force
```
Use the project number provided for your tutorial.
This step can be repeated at any time before submitting the case.


**#  Increase model throughput**

With the default task layout, this case runs relatively slowly, achieving a throughput of only about **4 model years per day**.

**Model throughput** measures how quickly the model runs. It is typically expressed in **model years simulated per wall-clock day** (or simply *years per day*). For example, a throughput of **4 years/day** means that if the model ran continuously for 24 hours, it would simulate 4 years of climate.

To improve throughput, change the task layout before running `case.setup`:

```bash
cd $CASEDIR
./xmlchange NTASKS=-6
```


**# Setup**

Invoke ``case.setup`` with the command:
```    
cd $CASEDIR
./case.setup    
```  

**# Customize namelists**

Edit the file ``user_nl_cam``and add the lines:

```
nhtfrq(2) = -3
mfilt(2) = 240
fincl2 = 'TS:I','PS:I', 'U850:I','T850:I','PRECT:I','LHFLX:I','SHFLX:I','FLNT:I','FLNS:I'
interpolate_output(2) = .true.,.true.
interpolate_nlat(2)   =  91,91
interpolate_nlon(2)   = 180,180
```
This creates a second CAM history stream, h1, with 3-hourly instantaneous output.

You can edit the file with a text editor. Alternatively, you can add the lines using `echo`:
```
echo "nhtfrq(2) = -3">> user_nl_cam    
echo "mfilt(2) = 240">> user_nl_cam
echo "fincl2 = 'TS:I','PS:I', 'U850:I','T850:I','PRECT:I','LHFLX:I','SHFLX:I','FLNT:I','FLNS:I'">> user_nl_cam
echo "interpolate_output(2) = .true.,.true.">> user_nl_cam    
echo "interpolate_nlat(2)   = 91,91">> user_nl_cam    
echo "interpolate_nlon(2)   = 180,180">> user_nl_cam    
echo "">> user_nl_cam
```


Build the namelists:
```
./preview_namelists
```
This step is optional because `preview_namelists` is automatically called by `case.build` and `case.submit`.

However, it is useful to check that your changes appear in:
```
$CASEDIR/CaseDocs/atm_in
```


**# Set run length**

If needed, change the ``run length``. If you want to run 5 days, you don't have to do this, as 5 days is the default. 
```   
./xmlchange STOP_N=5,STOP_OPTION=ndays
```


**# Build and submit**:
```
qcmd -- ./case.build
./case.submit
```

------------


**# Check your solution**

When the run is complete, examine the history files in the archive directory.
    
(1) Check that your archive directory on derecho (The path will be different on other machines): 
```
cd /glade/derecho/scratch/$USER/archive/$CASENAME/atm/hist
ls 
```

As your run is only 5-day, there should be no monthly file (``h0``). You should see 3-hourly ``h1`` files.

(2) Inspect the contents of the ``h1`` files using ``ncdump``.

```
ncdump -h fhist.control.cam.h1i.1979-01-01-10800.nc
```


- The file should contain the instantaneous output in the file ``h1`` for the variables:
```
 	float FLNS(time, lat, lon) ;
		FLNS:Sampling_Sequence = "rad_lwsw" ;
		FLNS:units = "W/m2" ;
		FLNS:long_name = "Net longwave flux at surface" ;
		FLNS:cell_methods = "time: point" ;
	float FLNT(time, lat, lon) ;
		FLNT:Sampling_Sequence = "rad_lwsw" ;
		FLNT:units = "W/m2" ;
		FLNT:long_name = "Net longwave flux at top of model" ;
		FLNT:cell_methods = "time: point" ;
	float LHFLX(time, lat, lon) ;
		LHFLX:units = "W/m2" ;
		LHFLX:long_name = "Surface latent heat flux" ;
		LHFLX:cell_methods = "time: point" ;
	float PRECT(time, lat, lon) ;
		PRECT:units = "m/s" ;
		PRECT:long_name = "Total (convective and large-scale) precipitation rate (liq + ice)" ;
		PRECT:cell_methods = "time: point" ;
	float PS(time, lat, lon) ;
		PS:units = "Pa" ;
		PS:long_name = "Surface pressure" ;
		PS:cell_methods = "time: point" ;
	float SHFLX(time, lat, lon) ;
		SHFLX:units = "W/m2" ;
		SHFLX:long_name = "Surface sensible heat flux" ;
		SHFLX:cell_methods = "time: point" ;
	float T850(time, lat, lon) ;
		T850:units = "K" ;
		T850:long_name = "Temperature at 850 mbar pressure surface" ;
		T850:cell_methods = "time: point" ;
	float TS(time, lat, lon) ;
		TS:units = "K" ;
		TS:long_name = "Surface temperature (radiative)" ;
		TS:cell_methods = "time: point" ;
	float U850(time, lat, lon) ;
		U850:units = "m/s" ;
		U850:long_name = "Zonal wind at 850 mbar pressure surface" ;
		U850:cell_methods = "time: point" ;
	float V850(time, lat, lon) ;
		V850:units = "m/s" ;
		V850:long_name = "Meridional wind at 850 mbar pressure surface" ;
		V850:cell_methods = "time: point" ;
```
Note that these variables have `cell_methods = "time: point" ` attribute which indicates the output is instantaneous. 

</details>
</div>

